# Importing Libraries

In [ ]:
from torch_pdegraph.utilities import *
import numpy as np
from matplotlib import image as mpimg
import faiss
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch
from skimage.util import view_as_windows
from matplotlib import image as mpimg

# Function to save the image

In [2]:
def saveImg(img, file_path):
    """
    Saves the image to the specified file path after normalizing it. 
    If the image is 3D with identical color channels, it converts it to grayscale before saving.
    
    param img: The input image, can be 2D or 3D array.
    param file_path: The path where to save the image, including the file name and extension.
    """
    # Check if img is a 2D grayscale or a 3D RGB where all channels are identical
    if len(img.shape) == 3 and np.all(img[:, :, 0] == img[:, :, 1]) and np.all(img[:, :, 1] == img[:, :, 2]):
        # If 3D RGB with identical channels, take the first channel to convert to grayscale
        img = img[:, :, 0]

    # Normalize the image to 0-1 range if it's not already in uint8 format
    if img.dtype != np.uint8:
        img = (img - np.min(img)) / (np.max(img) - np.min(img))

    # Save the image
    mpimg.imsave(file_path, img, cmap='gray')

# Load and process the image

In [ ]:
image_path = "../data/lena.png" # Input the correct image path here
x = mpimg.imread(image_path)
if x.ndim == 3:
    x = np.dot(x[..., :3], [0.2989, 0.5870, 0.1140])  # Convert to grayscale
x = addNoise(7,x)
dispImg(x)

# Create the graph

In [4]:
import numpy as np
import torch

# Image dimensions
rows, cols = x.shape

# Initialize lists for edges and weights
edge_list = []
weights = []

sigma = 0.4  # Define sigma for the weight function

# Iterate over each pixel in the image
for i in range(rows):
    for j in range(cols):
        center_value = x[i, j]
        neighbors = [(i-1, j), (i+1, j), (i, j-1), (i, j+1)]  # Up, Down, Left, Right

        for ni, nj in neighbors:
            # Check if neighbor is within bounds
            if 0 <= ni < rows and 0 <= nj < cols:
                neighbor_value = x[ni, nj]
                # Calculate absolute difference
                dist = abs(center_value - neighbor_value)
                # Compute weight based on absolute difference
                weight = np.exp(-dist/(sigma*sigma))

                # Calculate indices for the edge list
                center_idx = i * cols + j
                neighbor_idx = ni * cols + nj

                # Append to the edge list and weight list
                edge_list.append([center_idx, neighbor_idx])
                weights.append(weight)

# Convert edge list and weights to PyTorch tensors
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous().to('cuda:0')
edge_attr = torch.tensor(weights, dtype=torch.float).view(-1, 1).to('cuda:0')

# Assuming a Graph class is defined to take these as input
graph = Graph(edge_index, edge_attr)  # a custom graph object you've defined already somewhere

# Save the graph (if needed)

In [5]:
torch.save({'edge_attr':edge_attr, 'edge_index': edge_index},'../tensors/biggerGraph/tensorsmid.pt')

# Load the graph (if needed)

In [4]:
tensors = torch.load('../tensors/biggerGraph/tensorsmid.pt')
edge_attr = tensors['edge_attr']
edge_index = tensors['edge_index']
graph = Graph(edge_index, edge_attr)

# Run the iteration and display the images

In [ ]:
import importlib
import pLaplacianDiffusion

# After making changes to the pLaplacianDiffusion.py file
importlib.reload(pLaplacianDiffusion)

#hyperparams
hp = dict(x0=torch.tensor(toFmat(x),dtype=torch.float32).cuda(), lamb=1, alpha=0, beta=1)

itr = 20
features = torch.tensor(toFmat(x),dtype=torch.float32).cuda()

pde = pLaplacianDiffusion.PDE(graph, **hp)
new_features = pde(features, itr)
y = new_features.cpu().numpy()
y = toImg(y,x.shape)
z = mpimg.imread("../data/barbara.png") # input the correct image path here
dispImg(x)
dispImg(y)
dispImg(z)

# Save the output image

In [17]:
saveImg(y,"../output/35.png") # input the correct output path and filename